# Convolutional Neural Networks (CNNs)

## Created by Trung Nghi Tran

# Tailor Gemini: Building an Intelligent Customer Support System

# Introduction
This notebook shows how to build an custome customer support system based on Google's Gemini API. The system matches incoming customer queries with relevant historical support tickets and generates improved resolution responses. By using advanced language processing, it helps understand what the customer is asking, leading to more accurate and helpful responses.

->  The goal is to show how we can tailor the tone of responses based on historical customer support tickets.

# Why This Approach
This project uses Google's Gemini API rather than training a custom model from scratch for several important reasons:

- Hardware Limitations: Training large language models requires significant computational resources. Using a Windows laptop for such intensive training would be impractical due to memory and processing constraints.

- Cost Efficiency: Gemini offers a free API tier, making it accessible for development and testing without the financial barriers associated with platforms like OpenAI's GPT models.

- Pre-trained Intelligence: Gemini models have already been trained on massive datasets, providing sophisticated language understanding capabilities out-of-the-box.

- Embedding Support: The API provides text embedding functionality, allowing us to convert customer queries into mathematical vectors for semantic similarity matching without building complex machine learning infrastructure.

- Ease of Implementation: Using the API significantly reduces development time and complexity while still providing state-of-the-art NLP capabilities.
Scalability: This approach can easily scale from prototype to production without requiring significant changes to the underlying system architecture.

This API-based approach represents a practical balance between functionality, accessibility, and development effort - making advanced AI capabilities available even without specialized hardware or extensive machine learning expertise.

# About the Dataset

This project utilizes the "Customer Support Ticket Dataset" available on Kaggle (https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset). The dataset contains historical customer support tickets with the following key information:

- Ticket Description: The original query or problem reported by the customer
- Resolution: The solution or response provided by support agents
- Additional metadata: Information like ticket categories, priorities, and customer details

we will have an csv file in this datasheet

# API Source
The API used in this project is sourced from Google AI Studio, where I obtained access to the Gemini API. Google AI Studio allows users to explore and utilize their generative models to build powerful AI systems quickly, efficiently, and the most importantly it is free :)

- URL: https://aistudio.google.com/apikey
- Note: we need an google account to require Gemini's API

# Required Libraries and Installation

- pandas
- numpy
- scikit-learn
- matplotlib
- google-generativeai: Official Google library for accessing Gemini API

commnad in anaconda powershell: 

- conda install -c conda-forge pandas
- conda install -c conda-forge numpy
- conda install -c conda-forge scikit-learn
- conda install -c conda-forge matplotlib
- conda install google-generativeai


# Implementation 

let import all of the need libraries for this notebook

In [ ]:
# Setup the Gemini API
import google.generativeai as genai
import pandas as pd
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

let check what gemini version we can access via our free account 
please copy your API key To api_key, it is free on the website.

In [ ]:
# we will take the API key from the website and put it here
genai.configure(api_key="") 

# List available models
models = genai.list_models()
print("Available models:")
for model in models:
    print(f"- {model.name}")

that's a lot of version we can access (more options)

we will this version on this notebook: models/text-embedding-004

# Load Data

In [ ]:
# Load the dataset
df = pd.read_csv("customer_support_tickets.csv")  # Update with the correct filename

print(f"Loaded {len(df)} support tickets") # check the number of tickets loaded

# let check how our datasheet looks like
df.head()

# Define Embedding Function

this step allows the system to convert text into a numerical form that can be compared for similarity Without this step, the system would not be able to properly assess which tickets are similar to the user query.

for example:  "internet not working" and "connection problems" are related concepts even though they use different words.

In [ ]:
# Choose the model that will be used for embedding
embedding_model = "models/text-embedding-004"

# Function to generate embeddings 
def get_embedding(text):
    response = genai.embed_content(model=embedding_model, content=text)
    return np.array(response["embedding"])

# Test the embedding function
test_embedding = get_embedding("This is a test message")
print(f"Embedding shape: {test_embedding.shape}")

The number 768 is the size of the vector that represents the meaning of the text.  the model transforms the text into a list of 768 numbers (those numbers are the importatn information or meaning)

# Generate Embeddings for All Tickets

- this step is necessary to make the system capable of understanding customer queries in a meaningful way and providing relevant, accurate responses quickly.

- caution: this step take a lot of time (around 30 minutes)

Note: 
- The code loops through each support ticket's description and generates an embedding for each one.
- Once all embeddings are generated, it saves the updated dataset with embeddings as a file to avoid regenerating them later.

In [ ]:
# Track progress
total = len(df)
print(f"Generating embeddings for {total} tickets...") 
start_time = time.time()

# Create a new column for embeddings
df["embedding"] = None

# Process tickets in smaller batches to show progress
for i, row in df.iterrows():
    if i % 10 == 0:
        print(f"Processing {i}/{total} tickets ({i/total*100:.1f}%)...")
    df.at[i, "embedding"] = get_embedding(row["Ticket Description"])
    
# Save the embeddings to avoid recomputing in the future
df.to_pickle("customer_support_tickets_with_embeddings.pkl")
print(f"Embeddings generated and saved in {time.time() - start_time:.2f} seconds")

# Similarity Search Function

- This function helps to quickly find the best match between a new query and the historical tickets by comparing the "meanings" of the texts, not just the words.
- It ensures that the system can understand the intent behind the query and suggest the best possible solution from past tickets.

In [ ]:
# Function to find similar tickets
def find_similar_ticket(user_query):
    # Generate embedding for the query
    user_embedding = get_embedding(user_query)
    
    # Calculate similarity with all tickets
    df["similarity"] = df["embedding"].apply(lambda x: cosine_similarity([user_embedding], [x])[0][0])
    
    # Find the most similar ticket
    best_match = df.loc[df["similarity"].idxmax()]
    return best_match

# Test the function with a sample query
test_query = "My internet is not working"
best_match = find_similar_ticket(test_query)
print(f"Query: {test_query}")
print(f"Best matching ticket: {best_match['Ticket Description']}")
print(f"Resolution: {best_match['Resolution']}")
print(f"Similarity score: {best_match['similarity']:.4f}")

only 0.6311 it's not that good

# Improved Resolution Generation

This function takes the original response to a customer query, then improves it by making it sound nicer and easier to understand using AI. It ensures customers get the best possible experience from their support interaction.

the work flow: create a prompt with 3 parameters -> put it into AI to generate a better version -> output

In [ ]:
# Function to generate improved resolutions
def generate_improved_resolution(ticket_desc, original_resolution, user_query):
    prompt = f"""
    A customer asked: "{user_query}"
    
    The closest support ticket found was:
    Ticket Description: "{ticket_desc}"
    Previous Resolution: "{original_resolution}"

    Please rewrite the resolution in a clearer and more friendly way.
    Make sure to directly address their specific issue about "{user_query}".
    """

    # Use an appropriate Gemini model
    model = genai.GenerativeModel("gemini-1.5-flash-latest")
    response = model.generate_content(prompt)
    return response.text

# Test the function with our previous best match
improved_resolution = generate_improved_resolution(
    best_match["Ticket Description"],
    best_match["Resolution"],
    test_query
)

print("Original resolution:")
print(best_match["Resolution"])
print("\nImproved resolution:")
print(improved_resolution)

# Complete Query Processing Function

in this step we gonna put everything together to working on the incoming query from the customer (findfind_similar_ticket, generate_improved_resolution) and we test it with a query

In [ ]:
# Function to process a customer query from end to end
def process_customer_query(query):
    # Find similar ticket
    best_match = find_similar_ticket(query)
    
    # Generate improved resolution
    improved_resolution = generate_improved_resolution(
        best_match["Ticket Description"], 
        best_match["Resolution"],
        query
    )
    
    # Return results
    return {
        "query": query,
        "best_match_ticket": best_match["Ticket Description"],
        "original_resolution": best_match["Resolution"],
        "similarity_score": best_match["similarity"],
        "improved_resolution": improved_resolution
    }

# Test with a new query
new_query = "I can't login to my account"
result = process_customer_query(new_query)

print(f"Query: {result['query']}")
print(f"Best matching ticket: {result['best_match_ticket']}")
print(f"Similarity score: {result['similarity_score']:.4f}")
print(f"Original resolution: {result['original_resolution']}")
print(f"Improved resolution: {result['improved_resolution']}")

that's sound like AI assistance which is what we want :) 

# Evaluate System Performance

the work flow: split the dataset -> take the test tickets -> compare it with the actual and the predict -> store the scores

In [ ]:
# Split data for evaluation
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42) # 80% train, 20% test
print(f"Training on {len(train_df)} tickets, testing on {len(test_df)} tickets")

# We'll use a subset of test data for faster evaluation
test_subset = test_df.sample(min(20, len(test_df)), random_state=42)

results = []

# Test each query
for i, test_row in test_subset.iterrows():
    query = test_row["Ticket Description"]
    actual_resolution = test_row["Resolution"]
    
    # Find similar ticket using only training data
    train_df["similarity"] = train_df["embedding"].apply(
        lambda x: cosine_similarity([test_row["embedding"]], [x])[0][0]
    )
    
    best_match = train_df.loc[train_df["similarity"].idxmax()]
    predicted_resolution = best_match["Resolution"]
    similarity_score = best_match["similarity"]
    
    # Store results
    results.append({
        "query": query,
        "actual": actual_resolution,
        "predicted": predicted_resolution,
        "similarity": similarity_score,
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Calculate average similarity score
avg_similarity = results_df["similarity"].mean()
print(f"Average similarity score: {avg_similarity:.4f}")

0.9240 -> 92% that's really good 

# Visualize Performance

In [ ]:
# Plot top 5 and bottom 5 similarities
results_sorted = results_df.sort_values("similarity", ascending=False)
top5 = results_sorted.head(5)
bottom5 = results_sorted.tail(5)

plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plt.bar(range(len(top5)), top5["similarity"], color='green')
plt.title("Top 5 Most Similar Matches")
plt.ylabel("Similarity Score")
plt.xticks(range(len(top5)), [f"Query {i+1}" for i in range(len(top5))])

plt.subplot(2, 1, 2)
plt.bar(range(len(bottom5)), bottom5["similarity"], color='orange')
plt.title("Bottom 5 Least Similar Matches")
plt.ylabel("Similarity Score")
plt.xticks(range(len(bottom5)), [f"Query {i+1}" for i in range(len(bottom5))])

plt.tight_layout()
plt.show()

as we can see in the graph above, even the 5 least similar matches is around 80% -> which mean that the performance is so good

# Conclusion

This notebook provides a hands-on approach to learning how to fine-tune and work with a large language model (LLM) API. The main focus of this project is to demonstrate how we can use the Google Gemini API to tailor the tone and clarity of customer support responses based on a dataset of historical support tickets. And we can also using another datasheet in another fields as well